# 04 Abnormal returns and outcomes

For every country index this notebook computes log returns, the global factor and a market model
fitted on a window placed around every trading day. From these it derives:
- AR and SAR of every event row of `events.csv` (main labels and alternative day 0 of robustness sets);
- the normal days of each country and the country thresholds of a strong reaction;
- the event outcomes (strong reaction at the headline and secondary threshold) and pre-event volatility;
- eligibility of events after the estimation-window rules.

**Blind notebook.** Outcomes are computed and saved but never shown: nothing below prints a strong-reaction
flag, a rate, a mean |AR| or |SAR|, or vol_pre by group, by event type or by event, and the number of
positives is not printed at all. Only blind checks of the machinery are printed. Results are looked at in
notebooks 05 and 06.

In [1]:
import sys
from pathlib import Path

# On Nuvolos the kernel does not start in the notebook's folder: find code/ as 02_sample does.
roots = [Path.cwd(), Path("/files"), Path.home()]
code_dir = next(p for r in roots for p in [r, *r.glob("**/elections_project/code")] if (p / "helpers.py").exists())
sys.path.insert(0, str(code_dir))
print("Working in", code_dir)

import hashlib
import time

import numpy as np
import pandas as pd

from helpers import (AR_WIN, EST_WIN, EVENTS_FROM, EXTREME_RETURN, INVALID_CLOSES, MIN_EST_DAYS, MIN_GLOBAL,
                     NORMAL_EXCL,
                     OUTCOME_DAYS, POL_EXCL, PRE_VOL_WIN, RESID_DDOF, SAMPLE_END, SEED, TICKERS,
                     load_derived, load_prices, save_derived)
from analysis import (ANALYSIS_GROUPS, QUANTILES, apply_robustness_set, build_panels, event_outcomes,
                      in_invalid_span, load_robustness_sets, market_model, normal_days, predicted_return,
                      thresholds)

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)

Working in /files/mywork/elections_project/code


## Inputs
Prices are the frozen raw closes. `events.csv` and `political_dates.csv` come from notebook 03; the
political dates already include every parliamentary election of the full ParlGov table (also before
1997) and their trading day on the index calendar.

In [2]:
prices = load_prices()
political_dates = load_derived("political_dates", parse_dates=["date", "trading_day"])
events = load_derived("events", parse_dates=["source_date", "day0", "day1", "eligible_from"])
events["eligible"] = events["eligible"].astype(bool)
events["ineligible_reason"] = events["ineligible_reason"].fillna("")
print(len(events), "event rows,", len(political_dates), "political dates")

# windows must not overlap: estimation ends before the first stored AR and before the vol_pre window
assert EST_WIN[1] < AR_WIN[0] <= PRE_VOL_WIN[0] and PRE_VOL_WIN[1] < 0 <= min(OUTCOME_DAYS)
assert max(OUTCOME_DAYS) <= AR_WIN[1]

302 event rows, 1069 political dates


## a) Returns and b) global factor

**Returns.** Log returns between consecutive valid closes of each index, on the trading calendar of
`analysis.trading_calendar` (the same function as in 03, so day 0 is never a removed close). Removed closes:
closes repeated inside a run of zero returns, and closes listed in `INVALID_CLOSES` because they are not
the index (the Greek series from 19 December 2014 to 12 May 2015, checked against the official closes of
the Athens Exchange). A return spanning more than `LONG_GAP_DAYS` calendar days is a long-gap return: it
never enters an estimation window or the normal days. The return of 13 May 2015 in Greece spans the
removed closes, so it is a long-gap return; during the removed span Greece contributes nothing to the
global factor of the other countries, because its last valid close is too old. Returns above
`EXTREME_RETURN` in absolute value are listed and kept. Israel has two day classes: a Monday return spans
two sessions (Yahoo has no Sunday closes while TASE traded on Sunday); every other country has one class.

**Global factor.** For country c at observation t (previous observation s), the equal-weighted mean over
the other indices j of ln P(j, t*) − ln P(j, s*), where x* is j's last valid close at or before x. Index j
contributes only if both s* and t* are on or after its entry date (first date with
`GLOBAL_ENTRY_HISTORY` trading days of history) and its close at s* is at most `GLOBAL_MAX_STALE_DAYS`
calendar days old. With fewer than `MIN_GLOBAL` contributors the factor is missing.

**What the index manifest implies.** The indices are in local currency and their return type is mixed
(some gross of dividends, some price indices, many unverified). Nothing is converted: the global factor
stays the equal-weighted mean of local-currency log returns. Dividend differences are about 0.01% a day and
roughly constant, so they are absorbed by the intercept of the market model. Currency movements and very
volatile indices (for example XU100 in 1997-2001) make the global factor a noisier measure of world
markets: the model then removes less of the global component, abnormal returns are noisier and power is
lower. This does not bias the tests, because thresholds and the placebo are computed with the same model
on the same country's normal days.

In [3]:
panels = build_panels(prices, political_dates)

invalid = pd.Series({c: int(in_invalid_span(c, prices[TICKERS[c]].dropna().index).sum()) for c in TICKERS})
stale = pd.Series({c: prices[TICKERS[c]].notna().sum() - len(p) for c, p in panels.items()}) - invalid
extreme = pd.concat([p.loc[p["r"].abs() > EXTREME_RETURN, ["r"]].assign(country=c)
                     for c, p in panels.items()]).reset_index(names="date")
returns_summary = pd.DataFrame({
    "valid_closes": {c: len(p) for c, p in panels.items()},
    "invalid_removed": invalid,
    "stale_removed": stale,
    "extreme_returns": extreme.groupby("country").size(),
    "long_gap_returns": {c: int(p["long_gap"].sum()) for c, p in panels.items()},
}).fillna(0).astype(int)
print("INVALID_CLOSES:", INVALID_CLOSES)
print("Valid closes, closes removed (invalid data, stale), |r| above EXTREME_RETURN, long-gap returns:")
print(returns_summary.to_string())
print(f"\nEvery |r| > {EXTREME_RETURN} (kept):")
print(extreme[["country", "date", "r"]].round({"r": 4}).to_string(index=False))

INVALID_CLOSES: {'GRC': [('2014-12-19', '2015-05-12')]}
Valid closes, closes removed (invalid data, stale), |r| above EXTREME_RETURN, long-gap returns:
     valid_closes  invalid_removed  stale_removed  extreme_returns  long_gap_returns
AUS          8058                0              2                0                54
AUT          7927                0              0                0                66
BEL          8122                0              0                1                47
CAN          8012                0              0                0                22
CHE          8006                0              0                0                68
DEU          8082                0              0                0                58
DNK          2448                0              0                0                29
ESP          8084                0              0                1                50
EST          3336                0              9                0                2

In [4]:
# sd of daily log returns per index, in two periods (blind check of the manifest's warning)
split = pd.Timestamp("2003-01-01")
sd_period = pd.DataFrame({
    f"{EVENTS_FROM[:4]}-2002": {c: p.loc[(p.index >= EVENTS_FROM) & (p.index < split), "r"].std()
                               for c, p in panels.items()},
    f"2003-{SAMPLE_END[:7]}": {c: p.loc[(p.index >= split) & (p.index <= SAMPLE_END), "r"].std()
                               for c, p in panels.items()}})
print("sd of daily log returns (in %):")
print((100 * sd_period).round(2).to_string())

sd of daily log returns (in %):
     1997-2002  2003-2023-06
AUS       0.87          1.04
AUT       1.13          1.50
BEL       1.28          1.22
CAN       1.19          1.07
CHE       1.42          1.08
DEU       1.82          1.36
ESP       1.66          1.40
FRA       1.64          1.35
GBR       1.31          1.13
GRC       2.01          1.81
IRL       1.24          1.39
ISR       1.68          1.24
ITA       1.68          1.50
JPN       1.62          1.43
NLD       1.68          1.29
TUR       3.55          1.73
NZL        NaN          0.72
SWE        NaN          1.29
FIN        NaN          1.16
PRT        NaN          1.16
EST        NaN          0.77
LTU        NaN          0.63
DNK        NaN          1.11


In [5]:
# global factor: fewest contributors and share of observations with a missing factor, by year
# (the first close of each index has no previous observation, hence no return and no factor: left out)
gf_all = pd.concat([p.loc[p["r"].notna(), ["g", "n_contrib"]].assign(country=c) for c, p in panels.items()])
gf_all = gf_all[(gf_all.index >= EVENTS_FROM) & (gf_all.index <= SAMPLE_END)]
by_year = gf_all.groupby(gf_all.index.year).agg(min_contributors=("n_contrib", "min"),
                                                share_missing=("g", lambda g: g.isna().mean()))
print(f"Global factor by year (minimum MIN_GLOBAL = {MIN_GLOBAL}):")
print(by_year.round(4).to_string())

Global factor by year (minimum MIN_GLOBAL = 8):
      min_contributors  share_missing
date                                 
1997                11            0.0
1998                11            0.0
1999                14            0.0
2000                14            0.0
2001                14            0.0
2002                14            0.0
2003                14            0.0
2004                15            0.0
2005                15            0.0
2006                15            0.0
2007                15            0.0
2008                15            0.0
2009                16            0.0
2010                17            0.0
2011                17            0.0
2012                16            0.0
2013                16            0.0
2014                17            0.0
2015                20            0.0
2016                20            0.0
2017                21            0.0
2018                22            0.0
2019                21            0.0
20

## c) Market model
One function, `analysis.market_model`, fits r(t) = a + b0·g(t) + b1·g(t−1) by OLS, with g(t−1) the factor
of the country's previous observation. The window of a target day covers trading days `EST_WIN` relative to
it and keeps only days with a return, a factor and a lagged factor, that are not long-gap returns, are not
the day after a long-gap return, and are more than `POL_EXCL` trading days from every political date of
the country. The day after a long-gap return is left out because its lagged factor spans the whole gap
(for example five weeks after the Athens Exchange closure of 2015) and would enter the fit as an extreme
value of g(t−1); this exclusion was added before any outcome was looked at, and AR is still computed on
such days for events and normal days. The fit needs at least
`MIN_EST_DAYS` such days. The exclusion mask does not depend on the target day, so the fits of all days
come from rolling sums of masked cross-products (closed-form 3x3 normal equations); they are checked
against a direct least-squares fit below. The same fits serve the events (window around day 0) and the
normal days (window around each day).

In [6]:
start = time.perf_counter()
fits = {c: market_model(p) for c, p in panels.items()}
normal = normal_days(panels, fits)
runtime = time.perf_counter() - start
print(f"Market model for every trading day of every index + normal days: {runtime:.1f} s")

# no estimation day within +-POL_EXCL trading days of a political date (checked on the dates themselves)
for c, p in panels.items():
    pol = p.index.get_indexer(political_dates.loc[political_dates["country"] == c, "trading_day"].dropna())
    near = np.unique(np.clip((pol[:, None] + np.arange(-POL_EXCL, POL_EXCL + 1)).ravel(), 0, len(p) - 1))
    assert not p["est_ok"].to_numpy()[near].any(), c
    # no estimation day is a long-gap return or the day after one
    assert not (p["est_ok"] & (p["long_gap"] | p["long_gap"].shift(1, fill_value=False).astype(bool))).any(), c

Market model for every trading day of every index + normal days: 0.1 s


**Check of the fast fit.** 200 (country, day) pairs drawn with `SEED`, of which 20 are day 0 of eligible
events, are refitted directly with `np.linalg.lstsq` on the explicit list of window days. AR of the target
day, s, b0 and b1 must agree to 1e-8, and no window day may lie at relative position −19 or later.

In [7]:
rng = np.random.default_rng(SEED)
ev_ok = events[events["eligible"]]
pairs = [(c, int(t)) for c, t in ev_ok[["country", "day0_pos"]].sample(20, random_state=SEED).to_numpy()]
fitted = [(c, t) for c in TICKERS for t in np.flatnonzero(fits[c]["s"].notna().to_numpy())]
pairs += [fitted[i] for i in rng.choice(len(fitted), 200 - len(pairs), replace=False)]

for c, t in pairs:
    p, f = panels[c], fits[c].iloc[t]
    win = np.arange(max(t + EST_WIN[0], 0), t + EST_WIN[1] + 1)
    win = win[p["est_ok"].to_numpy()[win]]
    assert (win - t).max() <= EST_WIN[1] < AR_WIN[0] and len(win) == f["n_est"] >= MIN_EST_DAYS
    X = np.column_stack([np.ones(len(win)), p["g"].to_numpy()[win], p["g_lag"].to_numpy()[win]])
    y = p["r"].to_numpy()[win]
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    s_direct = np.sqrt(((y - X @ beta) ** 2).sum() / (len(win) - RESID_DDOF))
    ar_direct = p["r"].iloc[t] - (beta[0] + beta[1] * p["g"].iloc[t] + beta[2] * p["g_lag"].iloc[t])
    ar_fast = p["r"].iloc[t] - predicted_return(f, p["g"].iloc[t], p["g_lag"].iloc[t])
    assert np.allclose([f["a"], f["b0"], f["b1"], f["s"]], [*beta, s_direct], rtol=0, atol=1e-8), (c, t)
    assert np.isnan(ar_direct) and np.isnan(ar_fast) or abs(ar_direct - ar_fast) < 1e-8, (c, t)
print(f"Fast fit equals the direct fit on {len(pairs)} (country, day) pairs.")

Fast fit equals the direct fit on 200 (country, day) pairs.


## e) Normal days
For every trading day of each country from `EVENTS_FROM` to `SAMPLE_END`: the one-day AR with the window
placed around that day. A normal day has a valid AR, is not a long-gap return and is more than
`NORMAL_EXCL` trading days from every political date of the country.

## f) Thresholds
q(country, p, day class) is the p-quantile of |AR| over the normal days of that country and class, for
p = `P_MAIN` and `P_SEC`, computed with numpy's default quantile method (linear interpolation between the
two nearest order statistics).

In [8]:
pol_dist = pd.concat([p["pol_dist"].rename("d").to_frame().assign(country=c) for c, p in panels.items()])
nd = normal[normal["normal"]].merge(pol_dist.reset_index(names="date"), on=["country", "date"])
assert (nd["d"] > NORMAL_EXCL).all() and nd["ar"].notna().all()

thr = thresholds(normal)
q_main, q_sec = list(QUANTILES)
assert not thr.duplicated(["country", "day_class"]).any()
assert set(zip(thr["country"], thr["day_class"])) == set(zip(normal["country"], normal["day_class"]))
assert (thr[q_sec] < thr[q_main]).all()
print(thr.round(5).to_string(index=False))

country day_class  n_normal     q95     q90
    AUS       all      6362 0.01624 0.01264
    AUT       all      6132 0.01657 0.01258
    BEL       all      6306 0.01279 0.00971
    CAN       all      6385 0.01743 0.01304
    CHE       all      6427 0.01262 0.00999
    DEU       all      6490 0.01474 0.01138
    DNK       all      1371 0.01690 0.01368
    ESP       all      6455 0.01605 0.01228
    EST       all      2104 0.01344 0.01011
    FIN       all      2197 0.01150 0.00937
    FRA       all      6259 0.01228 0.00939
    GBR       all      6267 0.01167 0.00933
    GRC       all      5844 0.03170 0.02369
    IRL       all      6118 0.01745 0.01354
    ISR    monday      1035 0.02604 0.02199
    ISR     other      3221 0.01930 0.01480
    ITA       all      5802 0.01596 0.01243
    JPN       all      5829 0.02534 0.01993
    LTU       all      2174 0.01097 0.00798
    NLD       all      6413 0.01244 0.00961
    NZL       all      4624 0.01266 0.00996
    PRT       all      2300 0.01

### Blind checks of the normal days
- **Exceedance.** By construction about 1 − `P_MAIN` of normal days exceed q95 in each country. The share of
  pairs of consecutive normal days (d, d+1) with at least one day above q95 would be 1 − P_MAIN² (9.75%) if
  days were independent. Volatility clustering makes large days come together, so the pair share should be
  at or somewhat below 9.75%; a share far from it in one country would point to a problem with that index.
- **Market model.** b0 is the same-day loading on the global factor, b1 the loading on the previous
  observation's factor. Asia-Pacific indices close before Europe and America move, so part of the world
  news of day t reaches them on t+1: b1 should be larger for JPN, AUS and NZL than for the European indices.
- **Israel.** A Monday return spans two sessions, so its sd should be about √2 ≈ 1.4 times that of other
  days.

In [9]:
q95 = thr.set_index(["country", "day_class"])[q_main]
nm = normal.sort_values(["country", "date"]).copy()
nm["above"] = nm["ar"].abs() > q95.reindex(pd.MultiIndex.from_frame(nm[["country", "day_class"]])).to_numpy()
nxt = nm.groupby("country")[["normal", "above"]].shift(-1)
pair = nm["normal"] & nxt["normal"].eq(True)
exceed = pd.DataFrame({
    "normal_days": nm.groupby("country")["normal"].sum(),
    "share_above_q95": nm[nm["normal"]].groupby("country")["above"].mean(),
    "pairs": pair.groupby(nm["country"]).sum(),
    "share_pairs_with_one_above": (nm["above"] | nxt["above"].eq(True))[pair]
                                  .groupby(nm.loc[pair, "country"]).mean(),
})
print((exceed.assign(share_above_q95=100 * exceed["share_above_q95"],
                     share_pairs_with_one_above=100 * exceed["share_pairs_with_one_above"]))
      .round(2).to_string())

         normal_days  share_above_q95  pairs  share_pairs_with_one_above
country                                                                 
AUS             6362             5.01   6302                        8.62
AUT             6132             5.01   6059                        9.13
BEL             6306             5.01   6251                        8.85
CAN             6385             5.01   6355                        8.65
CHE             6427             5.01   6363                        9.15
DEU             6490             5.01   6431                        8.75
DNK             1371             5.03   1351                        9.47
ESP             6455             5.00   6404                        8.89
EST             2104             5.04   2081                        8.84
FIN             2197             5.01   2174                        9.11
FRA             6259             5.00   6202                        9.08
GBR             6267             5.01   6198       

In [10]:
coef = pd.concat([fits[c].loc[normal.loc[(normal["country"] == c) & normal["normal"], "date"],
                              ["b0", "b1", "r2"]].assign(country=c) for c in TICKERS])
print("Market model over normal days: 10th, 50th and 90th percentile of b0, b1 and R2")
print(coef.groupby("country").quantile([0.1, 0.5, 0.9]).unstack().round(3).to_string())

EUROPE = ["AUT", "BEL", "CHE", "DEU", "DNK", "ESP", "EST", "FIN", "FRA", "GBR", "GRC", "IRL", "ITA", "LTU",
          "NLD", "PRT", "SWE"]
mean_b1 = coef.groupby("country")["b1"].mean()
print("\nMean b1: " + ", ".join(f"{c} {mean_b1[c]:.3f}" for c in ["JPN", "AUS", "NZL"])
      + f" | European indices (mean of country means) {mean_b1[EUROPE].mean():.3f}")
if not (mean_b1[["JPN", "AUS", "NZL"]] > mean_b1[EUROPE].mean()).all():
    print("NOTE: b1 is not larger for every Asia-Pacific index than for Europe.")

isr = normal[(normal["country"] == "ISR") & normal["normal"]]
ratio = isr.loc[isr["day_class"] == "monday", "ar"].std() / isr.loc[isr["day_class"] == "other", "ar"].std()
print(f"\nIsrael: sd(AR) Monday / other days = {ratio:.2f}")

Market model over normal days: 10th, 50th and 90th percentile of b0, b1 and R2
            b0                   b1                   r2              
           0.1    0.5    0.9    0.1    0.5    0.9    0.1    0.5    0.9
country                                                               
AUS      0.191  0.407  0.588  0.189  0.328  0.443  0.187  0.285  0.411
AUT      0.385  1.263  1.478 -0.095  0.000  0.091  0.206  0.631  0.780
BEL      0.793  1.116  1.329 -0.168 -0.091  0.034  0.450  0.752  0.881
CAN      0.426  0.553  0.820 -0.132 -0.020  0.048  0.182  0.314  0.455
CHE      0.746  0.992  1.221 -0.186 -0.088  0.013  0.478  0.678  0.785
DEU      1.102  1.358  1.586 -0.261 -0.164 -0.060  0.614  0.765  0.865
DNK      0.711  0.969  1.112 -0.122 -0.066  0.048  0.283  0.441  0.665
ESP      1.106  1.277  1.592 -0.224 -0.134 -0.051  0.526  0.713  0.839
EST      0.079  0.248  0.547  0.006  0.076  0.153  0.024  0.103  0.316
FIN      1.062  1.249  1.355 -0.146 -0.071  0.101  0.581  0.740  0.84

## d) and g) Events: abnormal returns, eligibility and outcomes
For every row of `events.csv` (main labels and alternative day 0): AR and SAR = AR / s for every day of
`AR_WIN`, all outside the estimation window. An event eligible in 03 stays eligible if its window has at
least `MIN_EST_DAYS` valid days, day +1 exists, the global factor is defined on day 0 and day +1, and AR(0)
and AR(+1) can be computed (the lag term also needs the factor of day −1). Events whose day 0 or day +1 is a
long-gap return are kept and flagged.

Outcomes: strong = 1 if max(|AR(0)|, |AR(+1)|) exceeds the threshold, each day compared with the threshold
of its own day class, at p = `P_MAIN` (strong95) and p = `P_SEC` (strong90).
vol_pre = ln( sd of AR over `PRE_VOL_WIN` / sd_c ), with sd_c the sd of AR over all normal days of the
country; both sds use `VOL_DDOF`.

In [11]:
outcomes, event_ar, pre_missing = event_outcomes(events, panels, fits, normal, thr)
assert len(outcomes) == len(events)
ok = outcomes["eligible"]

# every event with an AR has enough estimation days and enough factor contributors on day 0 and day +1
for e in outcomes[ok].merge(events[["event_id", "variant", "day0_pos"]]).itertuples():
    nc = panels[e.country]["n_contrib"].to_numpy()[[int(e.day0_pos) + k for k in OUTCOME_DAYS]]
    assert e.n_est_days >= MIN_EST_DAYS and (nc >= MIN_GLOBAL).all(), e.event_id
assert set(event_ar["rel_day"]) == set(range(AR_WIN[0], AR_WIN[1] + 1))
assert event_ar["rel_day"].min() > EST_WIN[1]
assert outcomes.loc[~ok, [c for c in outcomes if c.startswith(("ar", "sar", "strong"))] + ["vol_pre"]].isna().all().all()

reason_04 = outcomes["drop_reason"].where(~outcomes["drop_reason"].str.startswith("not eligible in 03"), "")
print("Event rows:", len(outcomes), "| eligible after 03 and 04:", int(ok.sum()))
print("Dropped here (not by 03):", int((reason_04 != "").sum()))
print('Dropped with "AR(0) or AR(+1) not computable":',
      int((outcomes["drop_reason"] == "AR(0) or AR(+1) not computable").sum()))
print("Events whose day 0 or day +1 is a long-gap return (kept):",
      outcomes.loc[ok & outcomes["long_gap"], "event_id"].tolist())
print("\nDropped event rows and reasons:")
print(outcomes.loc[~ok, ["event_id", "variant", "drop_reason"]].to_string(index=False))

Event rows: 302 | eligible after 03 and 04: 267
Dropped here (not by 03): 0
Dropped with "AR(0) or AR(+1) not computable": 0
Events whose day 0 or day +1 is a long-gap return (kept): ['ISR_E718', 'ISR_E1090']

Dropped event rows and reasons:
event_id variant                                                           drop_reason
DNK_E389    main               not eligible in 03: fewer than 270 earlier trading days
DNK_E632    main               not eligible in 03: fewer than 270 earlier trading days
DNK_E563    main               not eligible in 03: fewer than 270 earlier trading days
DNK_E174    main               not eligible in 03: fewer than 270 earlier trading days
DNK_E722    main               not eligible in 03: fewer than 270 earlier trading days
DNK_E976    main               not eligible in 03: fewer than 270 earlier trading days
EST_E239    main               not eligible in 03: fewer than 270 earlier trading days
EST_E423    main               not eligible in 03: fewer than 

In [12]:
# vol_pre needs a valid AR on every day of PRE_VOL_WIN; any gap must be discussed before choosing a rule
if len(pre_missing):
    print("Eligible events with a missing AR inside PRE_VOL_WIN:")
    print(pre_missing.to_string(index=False))
assert pre_missing.empty, "Some eligible events lack AR inside PRE_VOL_WIN: stop and decide what to do."
assert outcomes.loc[ok, "vol_pre"].notna().all()

## h) Robustness sets after the eligibility of 03 and 04
`apply_robustness_set(events, set_name, outcomes)` keeps an event only if it is eligible in both 03 and 04,
then applies the overlap drop and the overlap10 flag among those events. Called without outcomes it must
reproduce the counts written by 03. Only counts are printed.

In [13]:
rob = load_robustness_sets()
SETS = ["main"] + list(dict.fromkeys(rob["set"]))
expected_crises = {"main": 80, "R_add": 88, "R_drop": 78, "R_vol": 73, "R_bundle": 82, "R_tv": 64,
                   "R_split": 79}
flow03 = load_derived("sample_flow").set_index(["set", "stage", "group"])["n"]
ok04 = outcomes.set_index(["event_id", "variant"])["eligible"]
rows = []
for s in SETS:
    t03 = apply_robustness_set(events, s)
    t = apply_robustness_set(events, s, outcomes)
    assert (t["group"] == "crisis").sum() == expected_crises[s]
    for g in ANALYSIS_GROUPS:
        assert (t03["group"] == g).sum() == flow03[(s, "coded", g)]
        assert ((t03["group"] == g) & t03["eligible"]).sum() == flow03[(s, "eligible", g)]
        assert ((t03["group"] == g) & t03["eligible"] & ~t03["overlap_drop"]).sum() == flow03[(s, "after overlap", g)]
    key = pd.MultiIndex.from_frame(t[["event_id", "variant"]])
    e03 = events.set_index(["event_id", "variant"])["eligible"].reindex(key).to_numpy()
    assert not (t["eligible"] & ~(e03 & ok04.reindex(key).to_numpy())).any()
    for g in ANALYSIS_GROUPS:
        rows.append({"set": s, "group": g, "eligible": int(((t["group"] == g) & t["eligible"]).sum()),
                     "after overlap": int(((t["group"] == g) & t["eligible"] & ~t["overlap_drop"]).sum()),
                     "overlap10": int(((t["group"] == g) & t["overlap10"]).sum())})
counts = pd.DataFrame(rows)
print("Event counts after eligibility in 03 and 04 (counts only):")
print(counts.pivot(index="group", columns="set", values=["eligible", "after overlap", "overlap10"])
      .reindex(columns=SETS, level=1).to_string())

Event counts after eligibility in 03 and 04 (counts only):
         eligible                                          after overlap                                          overlap10                                         
set          main R_add R_drop R_vol R_bundle R_tv R_split          main R_add R_drop R_vol R_bundle R_tv R_split      main R_add R_drop R_vol R_bundle R_tv R_split
group                                                                                                                                                               
crisis         76    83     74    70       78   60      75            76    83     74    70       78   60      75         0     0      0     0        0    0       0
election      144   144    144   144      144  144     144           144   144    144   144      144  144     144         0     0      0     0        0    0       0
planned        43    38     43    43       41   40      44            43    38     43    43       41   40      44   

## i) Outputs
- `returns.csv`: country, date, r, long_gap, day_class (full price history of each index).
- `global_factor.csv`: country, date, g, n_contrib.
- `normal_days.csv`: country, date, day_class, ar, normal (`EVENTS_FROM` to `SAMPLE_END`).
- `thresholds.csv`: one row per country and day class.
- `event_ar.csv`: AR and SAR of every eligible event row for each day of `AR_WIN`.
- `event_outcomes.csv`: one row per event row of `events.csv`.

In [14]:
returns = pd.concat([p[["r", "long_gap", "day_class"]].assign(country=c) for c, p in panels.items()])
returns = returns.reset_index(names="date")[["country", "date", "r", "long_gap", "day_class"]]
global_fac = pd.concat([p[["g", "n_contrib"]].assign(country=c) for c, p in panels.items()])
global_fac = global_fac.reset_index(names="date")
global_fac = global_fac[["country", "date", "g", "n_contrib"]]
for name, df in [("returns", returns), ("global_factor", global_fac), ("normal_days", normal),
                 ("thresholds", thr), ("event_ar", event_ar), ("event_outcomes", outcomes)]:
    print(save_derived(df, name), len(df), "rows")

/files/mywork/elections_project/data/derived/returns.csv 150449 rows
/files/mywork/elections_project/data/derived/global_factor.csv 150449 rows
/files/mywork/elections_project/data/derived/normal_days.csv 124993 rows
/files/mywork/elections_project/data/derived/thresholds.csv 24 rows
/files/mywork/elections_project/data/derived/event_ar.csv 8010 rows
/files/mywork/elections_project/data/derived/event_outcomes.csv 302 rows


## Reproducibility
The whole computation is run a second time from the raw prices; the two `event_outcomes` tables must be
byte-identical.

In [15]:
def outcomes_hash(df):
    return hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()

panels2 = build_panels(load_prices(), political_dates)
fits2 = {c: market_model(p) for c, p in panels2.items()}
normal2 = normal_days(panels2, fits2)
outcomes2, _, _ = event_outcomes(events, panels2, fits2, normal2, thresholds(normal2))
h1, h2 = outcomes_hash(outcomes), outcomes_hash(outcomes2)
assert h1 == h2, "two runs gave different event_outcomes"
print("event_outcomes hash (both runs):", h1)

event_outcomes hash (both runs): 9de6910343cf24a9dd0db80cb6a97a51dc36a08a4ac09b02e439c4bf3d3e6a54
